# Mathematical Basics - Part 2

This notebook covers the material in the last three chapters of the CWJSN Mathematical Basics section. Since much of this material is also covered in the Math_Foundations repo, I will only add interesting new stuff I learn along the way.

In [ ]:
# Set up Revise.jl for automatic code reloading
# This only needs to be run once at the beginning of your notebook session
using Revise

# Load the package 
using Calculus
using Latexify

## Inequalities and Logical Expressions Chapter

### Geometric Mean expressed in Julia

It is not clear why Verzani chose in CWJSN to use the expression `(a * b)^(1/2)` in the example about the geometric mean, rather than `sqrt(a * b)` as the code for the expression $g = \sqrt{ab}$. A related question is whether `sqrt(a * b) == (a * b)^(1/2)` is always `true`. I asked Claude to provide some insight on this.

The geometric mean of $n$ numbers is

$$
G = \left(x_1 x_2 \cdots x_n\right)^{1/n}, \qquad n = 2:\ G = (ab)^{1/2} = \sqrt{ab},
$$

so the power form shows the definition and generalizes by changing the exponent. In the same CWJSN code block, the next line uses `sqrt` for the quadratic mean, so the choice may be stylistic.

| Measured (Julia 1.13.2, $10^7$ values of `rand()`) | Result |
| --- | --- |
| `x^(1/2) != sqrt(x)` | about 1.05% of values |
| largest difference | 1 ulp (unit in the last place) |
| example $x$ | `0.7092888978478461` |
| `x^(1/2)` | `0.8421929101149249` |
| `sqrt(x)` (correctly rounded) | `0.842192910114925` |

As for whether the two different expressions are `==`, running a test returns `false` for some inputs. With `a = 0.7092888978478461` and `b = 1.0` (chosen so the product will be equal to `x` in the table above):

```julia
sqrt(a * b) == (a * b)^(1/2)   # false
sqrt(a * b)                    # 0.842192910114925
(a * b)^(1/2)                  # 0.8421929101149249
```

Over a million pairs from `rand(2)`, 9,650 gave `false`, about 1 in 100.

**Why the two results differ:**

- **The exponent isn't the cause.** `1/2` is exactly `0.5` in binary, so `(a*b)^(1/2)` really does ask for the power 0.5.
- **`sqrt` always returns the closest floating-point number to the true root.** The IEEE floating-point standard requires that of square root, and the processor has an instruction for it.
- **`^` with a `Float64` exponent uses a general power routine.** It is built to be accurate to within about 1 unit in the last place for any exponent, but it doesn't promise the closest number. About 1% of the time it lands one step away from what `sqrt` gives.

Both results are within 1 unit in the last place of the true root, so `sqrt(a * b) ≈ (a * b)^(1/2)` is always `true`. It's the same `==` versus `≈` lesson as in the Part 1 section "Overflow, rounding and `≈`": two different routes to the same real number can land on adjacent floats.